### Building a shower environment to get the temp right every time!
### 1. Import Dependencies

In [4]:
# import gym stuff
import gym
from gym import Env
from gym.spaces import Discrete, Box, Dict, Tuple, MultiBinary, MultiDiscrete

#import helpers
import os
import numpy as np
import random

#import stable baselines stuff
from stable_baselines3 import PPO
from stable_baselines3.common.vec_env import DummyVecEnv
from stable_baselines3.common.evaluation import evaluate_policy

### Types of spaces

* Note: You can add multiple spaces for Tuple and Dict --Grouping spaces

In [10]:
Discrete(3).sample()

0

In [12]:
Box(0, 1, shape=(3,3)).sample()

array([[0.19830433, 0.5533447 , 0.6759064 ],
       [0.73680794, 0.45807102, 0.6950739 ],
       [0.07709676, 0.46603528, 0.20975766]], dtype=float32)

In [29]:
Tuple((Discrete(3), Box(0,1, shape= (3,3)), MultiBinary(4))).sample()

(1,
 array([[0.7947663 , 0.23528342, 0.40641415],
        [0.8391698 , 0.517411  , 0.19150941],
        [0.51133126, 0.15425459, 0.55466896]], dtype=float32),
 array([1, 1, 0, 0], dtype=int8))

In [30]:
Dict({'height': Discrete(3), "speed":Box(0, 100, shape = (1,)), "color":MultiBinary(5)}).sample()

OrderedDict([('color', array([1, 1, 1, 0, 0], dtype=int8)),
             ('height', 0),
             ('speed', array([46.488506], dtype=float32))])

In [23]:
MultiBinary(4).sample()

array([0, 0, 1, 0], dtype=int8)

In [28]:
MultiDiscrete([5,2,2]).sample()

array([1, 0, 1])

# Build an Environment
- Build an agent to give us the best shower possible
- Randomly temparature
- 37 and 39 degrees

In [60]:
class ShowerEnv(Env):
    def __init__(self):
        self.action_space = Discrete(3)
        self.observation_space = Box(low = 0, high=100, shape = (1,))
        self.state = 38 + random.randint(-3,3)
        self.shower_length = 60
        
    def step(self, action):
        # apply temp adj
        self.state += action-1

        #Decrease shower time
        self.shower_length -= 1

        #calc reward
        if self.state >= 37 and self.state <=39:
            reward = 1
        else:
            reward = -1
        
        if self.shower_length <=0:
            done = True
        else:
            done = False

        info= {}

        return self.state, reward, done, info
            

    def render(self):
        pass
    
    def reset(self):
        self.state = np.array([38 + random.randint(-3,3)]).astype(float)
        self.shower_length = 60
        return self.state

In [61]:
env = ShowerEnv()

In [37]:
env.observation_space.sample()

array([36.296917], dtype=float32)

In [38]:
env.action_space.sample()

0

### 2. Test Environment

In [43]:
episodes = 5
for episode in range(1, episodes + 1):
    obs = env.reset()
    done = False
    score = 0
    
    while not done:
        env.render()  # This shows the environment window if using render_mode="human"  # render func allows us to view the the graphical representation of the env.
        action = env.action_space.sample()  #Now we are using our saved model.
        obs, reward, done, info = env.step(action)
        score += reward
    print("Episodes:{} score:{}".format(episode, score))
env.close()

Episodes:1 score:-50
Episodes:2 score:-40
Episodes:3 score:-52
Episodes:4 score:-50
Episodes:5 score:-18


# Train Model

In [50]:
import shimmy

In [51]:
log_path = os.path.join("Training", "Logs")
model = PPO("MlpPolicy", env, verbose =1, tensorboard_log= log_path)

Using cpu device
Wrapping the env with a `Monitor` wrapper
Wrapping the env in a DummyVecEnv.


C:\Users\nmokkapa\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\stable_baselines3\common\vec_env\patch_gym.py:49: UserWarning: You provided an OpenAI Gym environment. We strongly recommend transitioning to Gymnasium environments. Stable-Baselines3 is automatically wrapping your environments in a compatibility layer, which could potentially cause issues.
  warnings.warn(


In [53]:
model.learn(total_timesteps= 100000)

Logging to Training\Logs\PPO_14
---------------------------------
| rollout/           |          |
|    ep_len_mean     | 60       |
|    ep_rew_mean     | -28.4    |
| time/              |          |
|    fps             | 1885     |
|    iterations      | 1        |
|    time_elapsed    | 1        |
|    total_timesteps | 2048     |
---------------------------------
-----------------------------------------
| rollout/                |             |
|    ep_len_mean          | 60          |
|    ep_rew_mean          | -27.3       |
| time/                   |             |
|    fps                  | 1266        |
|    iterations           | 2           |
|    time_elapsed         | 3           |
|    total_timesteps      | 4096        |
| train/                  |             |
|    approx_kl            | 0.006732082 |
|    clip_fraction        | 0.0704      |
|    clip_range           | 0.2         |
|    entropy_loss         | -1.09       |
|    explained_variance   | 7.03e-06    

### 4. Save and Reload Model

In [54]:
shower_path = os.path.join("Training", "saved Models", "Shower_Model_PPO")

In [55]:
model.save(shower_path)

In [56]:
del model

In [62]:
model = PPO.load(shower_path, env)

Wrapping the env with a `Monitor` wrapper
Wrapping the env in a DummyVecEnv.


C:\Users\nmokkapa\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\stable_baselines3\common\vec_env\patch_gym.py:49: UserWarning: You provided an OpenAI Gym environment. We strongly recommend transitioning to Gymnasium environments. Stable-Baselines3 is automatically wrapping your environments in a compatibility layer, which could potentially cause issues.
  warnings.warn(


### 5. Evaluate and Test

In [63]:
evaluate_policy(model, env, n_eval_episodes= 10, render = False)

C:\Users\nmokkapa\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\stable_baselines3\common\evaluation.py:67: UserWarning: Evaluation environment is not wrapped with a ``Monitor`` wrapper. This may result in reporting modified episode lengths and rewards, if other wrappers happen to modify these. Consider wrapping environment first with ``Monitor`` wrapper.
  warnings.warn(


(np.float64(59.4), np.float64(0.9165151389911681))